# EX_08 — Introducción a agentes (ejercicios)

**Notebook de referencia:** `notebook/08_Introduccion_Agentes.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Definir 2 herramientas

Escribe funciones Python puras `get_time_utc()` (puede ser fake) y `hash_text(s: str)` (usa `hashlib.sha256` en hex). Estas serán tus "tools".


In [3]:
# ==========================================
# FUNCION ANTERIOR (Línea 5)
# ==========================================
def funcion_previa_del_notebook():
    # Dejamos un 'pass' por si esta función estaba vacía
    # y así evitar el error de IndentationError
    pass

# ==========================================
# EJERCICIO: FUNCIÓN HASH_TEXT (Línea 8)
# ==========================================
def hash_text(s: str) -> str:
    """Genera un hash SHA-256 a partir de una cadena de texto."""
    import hashlib

    # 1. Convertimos el texto a bytes usando encode()
    texto_en_bytes = s.encode('utf-8')

    # 2. Calculamos el hash SHA-256
    hash_objeto = hashlib.sha256(texto_en_bytes)

    # 3. Devolvemos el hash en formato hexadecimal (string de texto)
    return hash_objeto.hexdigest()

# ==========================================
# PRUEBA DE LA FUNCIÓN
# ==========================================
# Probamos que funcione correctamente con un texto de ejemplo
print("Hash de 'hola':", hash_text("hola"))

Hash de 'hola': b221d9dbb083a7f33428d7c2a3c3198ae925614d70210e28716ccaa7cd4ddb79


## Actividad 2 — Cuándo usar tool

Para cada intención del usuario (`"What time is it?"`, `"Digest of hello"`), escribe en comentarios si el LLM debería llamar tool o responder directo.


In [4]:
# =====================================================================
# ACTIVIDAD 2: CUÁNDO USAR TOOL vs RESPUESTA DIRECTA
# =====================================================================

# Intención 1: "What time is it?"
# ---------------------------------------------------------------------
# LLM debería llamar TOOL.
# Razón: Los LLMs no tienen un reloj interno ni acceso al tiempo real del
# sistema. Para dar la hora exacta actual, necesitan una herramienta (como una
# función que ejecute 'datetime.now()').


# Intención 2: "Digest of hello"
# ---------------------------------------------------------------------
# LLM debería llamar TOOL.
# Razón: Aunque un LLM potente podría intentar emular un hash o conocer el
# hash de palabras comunes, los LLMs son malos con la manipulación exacta de
# bytes y caracteres a nivel criptográfico. Para garantizar que el hash
# SHA-256 (o MD5) de "hello" sea 100% correcto, debe delegar la tarea a la
# función 'hash_text' que creamos en el ejercicio anterior.


## Actividad 3 — Bucles

En español (celda markdown), explica el riesgo de **bucles infinitos** tool→modelo→tool y una mitigación (límite de pasos, detector de repetición).


_Tu explicación:_

...


In [ ]:
### ⚠️ El Riesgo de Bucles Infinitos (Tool ↔ Modelo) y cómo Mitigarlo

En la arquitectura de agentes, los modelos LLM deciden dinámicamente si necesitan usar una herramienta (*tool*) basándose en la pregunta del usuario. Aunque este sistema es muy potente, introduce un riesgo crítico de **bucle infinito (loop)**.

#### 🚨 ¿En qué consiste el riesgo?
Un bucle infinito ocurre cuando el modelo entra en un ciclo de razonamiento erróneo del cual no puede salir por sí mismo. El flujo se vuelve destructivo de la siguiente manera:
1. El **Modelo** decide llamar a una **Tool**.
2. La **Tool** ejecuta la acción y devuelve un resultado (u observación).
3. El **Modelo** malinterpreta el resultado, se confunde, o considera que no es suficiente, y decide volver a llamar a la *misma herramienta* (o a otra) con los mismos parámetros.
4. La **Tool** vuelve a responder exactamente lo mismo... repitiendo el ciclo indefinidamente.

**Consecuencias:** * Consumo masivo y absurdo de tokens de la API (lo que se traduce en dinero desperdiciado en segundos).
* Bloqueo del entorno de ejecución o congelamiento de la aplicación para el usuario final.

---

#### 🛡️ Estrategias de Mitigación

Para evitar que nuestro agente "se vuelva loco" gastando recursos, debemos implementar salvaguardas directas en el bucle de ejecución:

##### 1. Límite máximo de pasos (Max Iterations / Step Limit)
Es la solución más robusta e indispensable. Consiste en configurar un **contador rígido** en el bucle del agente (por ejemplo, un máximo de 5 o 10 iteraciones).
* *Cómo funciona:* Cada vez que el modelo decide usar una herramienta, el contador suma 1. Si el agente llega al límite fijado sin haber encontrado la `Respuesta Final`, el sistema detiene la ejecución a la fuerza de manera segura, lanzando una excepción o devolviendo un mensaje de error controlado.

##### 2. Detector de repetición (Repetition Detector)
Consiste en monitorizar el historial inmediato de las acciones del agente.
* *Cómo funciona:* El sistema almacena los últimos pasos (`Action` y `Action Input`). Si el detector identifica que el modelo está llamando a la misma herramienta con los mismos argumentos exactos por tercera vez consecutiva (o si detecta que la observación de la herramienta no cambia), intercepta el flujo. El sistema altera el prompt internamente para decirle al modelo: *"Ya has intentado esto y falló, cambia de estrategia o finaliza"* o, directamente, detiene el proceso.